### Calculating Tilt for mooring applications

In [1]:
# Calculating Tilt for mooring applications

In [2]:
# ------------------------------------------------------------
# Imports

import numpy as np
import xarray as xr
import os
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import glob
import gsw
import datetime
import pandas as pd

# End imports
# ------------------------------------------------------------


In [3]:
# ------------------------------------------------------------
# Definitions

def compute_orientation(df):
    """
    Calculate orientation angles from accelerometer and magnetometer data.
    Returns proper tilt angles in range -180° to +180° for mooring analysis.
    """
    Ax = df["Ax (g)"].to_numpy()
    Ay = df["Ay (g)"].to_numpy()
    Az = df["Az (g)"].to_numpy()
    Mx = df["Mx (mG)"].to_numpy()
    My = df["My (mG)"].to_numpy()
    Mz = df["Mz (mG)"].to_numpy()

    # Pitch and Roll from accelerometer
    pitch = np.arctan2(-Ax, np.sqrt(Ay**2 + Az**2))
    roll = np.arctan2(Ay, Az)

    # Yaw estimate from magnetometer and pitch/roll
    mag_x = Mx * np.cos(pitch) + Mz * np.sin(pitch)
    mag_y = Mx * np.sin(roll) * np.sin(pitch) + My * np.cos(roll) - Mz * np.sin(roll) * np.cos(pitch)
    yaw = np.arctan2(-mag_y, mag_x)

    # Convert radians to degrees - keep in proper -180 to +180 range for tilt analysis
    df["Pitch"] = np.degrees(pitch)
    df["Roll"] = np.degrees(roll) 
    df["Yaw"] = np.degrees(yaw)

    return df

# End Definitions
# ------------------------------------------------------------

In [18]:
# ------------------------------------------------------------
# Start Main
# ------------------------------------------------------------

# Configuration parameters
inst_type = 'MAT1'
processing_ver = 2
fgen = '/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/data_in/rec_202603/BASJAS_PTSUVW_202603'
# '/datasets/work/oa-aapp-ocean/work/2023_12_FOCUS/mooring/proc_1/rec_202504'

# Find all folders starting with MAT1
list_folders = sorted([d for d in os.listdir(fgen) 
                      if os.path.isdir(os.path.join(fgen, d)) and d.startswith('MAT1')])
print(f"Found {len(list_folders)} MAT1 folders: {list_folders}\n")

# Process each MAT1 folder
for f in list_folders:
    print(f"Processing folder: {f}")
    folder = os.path.join(fgen, f)
    os.chdir(folder)
    
    # Find *_AccellMag CSV files - try multiple search patterns
    # filist = sorted(glob.glob("*/*_AccelMag.csv"))  # One level deep
    filist = sorted(glob.glob("**/*_AccelMag.csv", recursive=True))  # Any depth
    
    file_in = filist[0]  # Process first file found
    print(f"  ✓ Loading: {file_in}")
    
    # Read accelerometer data
    df = pd.read_csv(file_in, parse_dates=["ISO 8601 Time"])
    df.rename(columns={"ISO 8601 Time": "Time"}, inplace=True)
    
    print(f"  Loaded {len(df):,} rows")
    print(f"  Time range: {df['Time'].min()} to {df['Time'].max()}\n")

print("Data preview:")
print(df.head())

In [ ]:
# # ------------------------------------------------------------
# # Interactive instrument selection using the reusable module
# # ------------------------------------------------------------

# # Import the instrument selector module
# import sys
# sys.path.append('/home/par520/w/mooring_proc')
# from instrument_selector import select_instrument

# # Interactive selection
# print("=== INSTRUMENT SELECTION FOR TILT ANALYSIS ===")
# selection = select_instrument()

# # Extract the selected information
# selected_location = selection['location']
# selected_instrument = selection['instrument_type'] 
# selected_serial_number = selection['serial_number']
# site_metadata = selection['metadata']
# file_in = selection['input_file']
# file_source = selection.get('file_source', 'unknown')
# file_available = selection.get('file_available', False)

# # Check if we have a valid file path
# if not file_available or file_in == 'FILE_PATH_NOT_AVAILABLE':
#     print(f"❌ ERROR: No file path available in metadata for this instrument")
#     print(f"   Location: {selected_location}")
#     print(f"   Instrument: {selected_instrument}")
#     print(f"   Serial: {selected_serial_number}")
#     print("\n📋 Available metadata columns for this entry:")
#     for col in ['data_in_path', 'data_in_file', 'proc_1_path', 'proc_1_file', 'proc_2_path', 'proc_2_file']:
#         val = site_metadata.get(col, 'N/A')
#         print(f"   {col}: {val}")
#     print("\n💡 You may need to:")
#     print("   1. Update the metadata CSV file with correct file paths")
#     print("   2. Use a different instrument/deployment")
#     print("   3. Manually specify the file path")
#     raise FileNotFoundError(f"No file path available in metadata")

# # Check if the file actually exists
# if not os.path.exists(file_in):
#     print(f"❌ ERROR: Input file not found: {file_in}")
#     print(f"   Source: {file_source} level in metadata")
#     print("💡 File path exists in metadata but file is not accessible.")
#     print("   Please check:")
#     print("   1. File path is correct")
#     print("   2. File permissions")
#     print("   3. Network/mount points if on remote storage")
#     raise FileNotFoundError(f"Input file not found: {file_in}")

# print(f"✅ Input file confirmed: {file_in}")
# print(f"📂 File source: {file_source} level")

# # Extract site code for naming
# site = selected_location.replace('BAS', '')  # Remove 'BAS' prefix if present
# pf_code = f"BAS{site}"

# print(f"📊 Processing {selected_instrument} {selected_serial_number} from {selected_location}")
# print(f"📁 File: {os.path.basename(file_in)}")

# # ------------------------------------------------------------
# # Load and prepare the data
# # ------------------------------------------------------------

# print(f"\n=== LOADING DATA ===")

# # Read the accelerometer data file 
# # Expecting CSV format with "ISO 8601 Time" column and accelerometer/magnetometer data
# try:
#     df = pd.read_csv(file_in, parse_dates=["ISO 8601 Time"])
#     print(f"✅ Data loaded successfully: {len(df):,} rows")
# except Exception as e:
#     print(f"❌ Error loading data: {e}")
#     print(f"💡 The file may not be in the expected format.")
#     print(f"   Expected: CSV with 'ISO 8601 Time' column")
#     print(f"   File: {file_in}")
#     raise

# # Rename the timestamp column for convenience
# df.rename(columns={"ISO 8601 Time": "Time"}, inplace=True)

# print("📋 Data overview:")
# print(f"  • Time range: {df['Time'].min()} to {df['Time'].max()}")
# print(f"  • Duration: {(df['Time'].max() - df['Time'].min()).days} days")
# print(f"  • Columns: {list(df.columns)}")
# print("\nFirst few rows:")
# print(df.head())

In [19]:
# compute the orientations
df = compute_orientation(df)
print(f"Data shape after orientation calculation: {df.shape}")
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# # Optional
# # Remove outliers from accelerometer data
# print("=== Removing outliers from accelerometer data ===")
# accel_columns = ['Ax (g)', 'Ay (g)', 'Az (g)', 'Mx (mG)', 'My (mG)', 'Mz (mG)']
# df_accel_clean, accel_outlier_info = remove_outliers(df, accel_columns, std_threshold=5)

# # Recompute orientations after outlier removal
# print("\n=== Recomputing orientations after outlier removal ===")
# df_accel_clean = compute_orientation(df_accel_clean)

# print(f"\nAccelerometer data shape after outlier removal: {df_accel_clean.shape}")
# print("First few rows after cleaning:")
# print(df_accel_clean[['Time'] + accel_columns + ['Pitch', 'Roll', 'Yaw']].head())

In [21]:
# ------------------------------------------------------------
# Time filtering setup
# ------------------------------------------------------------

# Get data time range
data_start = df['Time'].min()
data_end = df['Time'].max()
data_duration = (data_end - data_start).days

print(f" Available data time range:")
print(f"   Start: {data_start}")
print(f"   End: {data_end}")
print(f"   Duration: {data_duration} days")

time_start = pd.to_datetime("2024-07-31T05:20:00")  # Your specified start time
time_end = pd.to_datetime("2025-08-23T04:10:00")    # Your specified end time


# # Add some buffer time if using full data range (to avoid edge effects)
# if time_start == data_start and time_end == data_end and data_duration > 1:
#     buffer_hours = min(6, data_duration * 24 * 0.01)  # 1% of duration or 6 hours, whichever is smaller
#     time_start = time_start + pd.Timedelta(hours=buffer_hours)
#     time_end = time_end - pd.Timedelta(hours=buffer_hours)

print(f"   Start: {time_start}")
print(f"   End: {time_end}")
print(f"   Duration: {(time_end - time_start).days} days")

In [23]:
# ------------------------------------------------------------
# Tilt Analysis
# ------------------------------------------------------------

# Apply time filtering to the data
df_accel_filtered = df[(df["Time"] >= time_start) & (df["Time"] <= time_end)]
print(f"Time filter applied: {time_start} to {time_end}")
print(f"Filtered accelerometer data: {len(df_accel_filtered):,} rows\n")

# Calculate overall mooring tilt angle (layover angle)
mooring_tilt_angle = np.sqrt(df_accel_filtered['Pitch']**2 + df_accel_filtered['Roll']**2)

# Calculate tilt direction (azimuth of tilt vector)
tilt_azimuth = np.degrees(np.arctan2(df_accel_filtered['Roll'], df_accel_filtered['Pitch']))

# Normalize to 0-360 degrees for compass direction
tilt_azimuth = (tilt_azimuth + 360) % 360

print(f"Mooring Tilt Statistics:")
print(f"  Mean tilt angle: {mooring_tilt_angle.mean():.3f}°")
print(f"  Standard deviation: {mooring_tilt_angle.std():.3f}°")
print(f"  Maximum tilt angle: {mooring_tilt_angle.max():.3f}°")

print(f"\nTilt Direction Statistics:")
print(f"  Mean tilt direction: {tilt_azimuth.mean():.1f}° (from North)")

print(f"\nPitch and Roll Statistics:")
print(f"  Pitch: {df_accel_filtered['Pitch'].mean():.3f}° ± {df_accel_filtered['Pitch'].std():.3f}°")
print(f"  Roll: {df_accel_filtered['Roll'].mean():.3f}° ± {df_accel_filtered['Roll'].std():.3f}°")
print(f"  Max Pitch: {df_accel_filtered['Pitch'].max():.3f}°")
print(f"  Max Roll: {df_accel_filtered['Roll'].max():.3f}°")


In [24]:
# # ------------------------------------------------------------
# # Displacement Analysis with Correct Tilt Angles
# # ------------------------------------------------------------


# # Mooring configuration
# water_depth = 55.0  # meters
# seabed_height = 55.0  # meters
# instrument_start_depth = 17.0  # meters below sea level

# # Instrument heights above tilt sensor (your configuration)
# instrument_heights = {
#     'Tilt_sensor': 0.0,     # Reference point
#     'Inst_1m_a': 1.0,       # 1m above tilt sensor
#     'Inst_1m_b': 1.0,       # Another at 1m
#     'Inst_1m_c': 1.0,       # Another at 1m  
#     'Inst_6m': 6.0,         # 6m above tilt sensor
#     'Inst_18m': 18.0,       # 18m above tilt sensor
#     'Top_inst': 31.0        # Top instrument 31m above tilt sensor
# }

# print(f"Mooring Configuration:")
# print(f"  Water depth: {water_depth}m")
# print(f"  Tilt sensor depth: {instrument_start_depth}m below surface")
# print(f"  Instrument heights above tilt sensor:")
# for name, height in instrument_heights.items():
#     depth_below_surface = instrument_start_depth - height
#     print(f"    {name}: {height}m above sensor ({depth_below_surface:.1f}m depth)")

# def calculate_displacement(pitch_deg, roll_deg, height):
#     """Calculate horizontal displacement due to tilt."""
#     pitch_rad = np.radians(pitch_deg)
#     roll_rad = np.radians(roll_deg)
    
#     # Small angle approximation for typical mooring tilts
#     dx = height * np.sin(roll_rad)   # East displacement
#     dy = height * np.sin(pitch_rad)  # North displacement  
#     dz = height * (1 - np.cos(pitch_rad) * np.cos(roll_rad))  # Vertical (small)
    
#     horizontal_displacement = np.sqrt(dx**2 + dy**2)
    
#     return dx, dy, dz, horizontal_displacement

# # Calculate displacements for all instruments
# displacement_results = {}
# for inst_name, height in instrument_heights.items():
#     dx, dy, dz, horiz_disp = calculate_displacement(
#         df_accel_filtered['Pitch'], 
#         df_accel_filtered['Roll'], 
#         height
#     )
    
#     displacement_results[inst_name] = {
#         'dx': dx, 'dy': dy, 'dz': dz,
#         'horizontal_displacement': horiz_disp,
#         'height': height
#     }

# # Calculate maximum displacements
# print(f"\nMaximum Horizontal Displacements:")
# max_displacements = {}
# for inst_name, data in displacement_results.items():
#     if data['height'] > 0:
#         max_disp = data['horizontal_displacement'].max()
#         max_displacements[inst_name] = max_disp
#         depth = instrument_start_depth - data['height']
#         print(f"  {inst_name}: {max_disp:.4f}m (at {depth:.1f}m depth)")

# max_overall_disp = max(max_displacements.values())
# print(f"Maximum displacement: {max_overall_disp:.4f}m")
# print(f"Maximum tilt: {mooring_tilt_angle.max():.3f}°")

# print(f"\nTilt correction values:")
# print(f"  Mean pitch correction: {df_accel_filtered['Pitch'].mean():.4f}°")
# print(f"  Mean roll correction: {df_accel_filtered['Roll'].mean():.4f}°")
# print(f"  RMS tilt magnitude: {np.sqrt(np.mean(mooring_tilt_angle**2)):.4f}°")

In [27]:
# Clean 3D orientation plot with correct angles
print("Creating 3D orientation plots with correct tilt angles...\n")

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(15, 12))

# 3D scatter plot
ax1 = fig.add_subplot(221, projection='3d')
step = max(1, len(df_accel_filtered) // 1000)  # Sample for performance
df_sample = df_accel_filtered.iloc[::step]

scatter = ax1.scatter(df_sample["Pitch"], df_sample["Roll"], df_sample["Yaw"], 
                     c=range(len(df_sample)), cmap='viridis', alpha=0.7, s=15)
ax1.set_xlabel('Pitch (°)')
ax1.set_ylabel('Roll (°)')
ax1.set_zlabel('Yaw (°)')
ax1.set_title('3D Orientation Space\n(Corrected Angles)')

# Tilt angle over time
ax2 = fig.add_subplot(222)
ax2.plot(df_accel_filtered['Time'], mooring_tilt_angle, 'b-', alpha=0.7, linewidth=0.8)
ax2.axhline(y=mooring_tilt_angle.mean(), color='red', linestyle='--', 
           label=f'Mean: {mooring_tilt_angle.mean():.3f}°', alpha=0.8)
ax2.axhline(y=mooring_tilt_angle.quantile(0.95), color='orange', linestyle='--', 
           label=f'95th %ile: {mooring_tilt_angle.quantile(0.95):.3f}°', alpha=0.8)
ax2.set_title('Mooring Tilt Angle Over Time')
ax2.set_ylabel('Tilt Angle (°)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Displacement profile
ax3 = fig.add_subplot(223)
heights = [data['height'] for data in displacement_results.values()]
mean_displacements = [data['horizontal_displacement'].mean() for data in displacement_results.values()]
max_displacements_plot = [data['horizontal_displacement'].max() for data in displacement_results.values()]

ax3.plot(mean_displacements, heights, 'o-', color='blue', markersize=8, 
         linewidth=2, label='Mean displacement')
ax3.plot(max_displacements_plot, heights, 's-', color='red', markersize=6, 
         linewidth=2, label='Max displacement', alpha=0.7)

# Add instrument labels
inst_names = list(instrument_heights.keys())
for name, height, mean_disp in zip(inst_names, heights, mean_displacements):
    ax3.annotate(name, (mean_disp, height), xytext=(5, 0), 
                textcoords='offset points', fontsize=9, alpha=0.8)

# Add vertical line for mooring without tilt (zero displacement)
ax3.axvline(x=0, color='black', linestyle='-', linewidth=2, alpha=0.8, label='No tilt (vertical mooring)')

ax3.set_title('Displacement vs Height Above Tilt Sensor')
ax3.set_xlabel('Horizontal Displacement (m)')
ax3.set_ylabel('Height Above Tilt Sensor (m)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Add threshold lines
for threshold, label, color in [(0.1, '0.1m', 'gray'), (1.0, '1.0m (CTD)', 'orange'), 
                               (2.0, '2.0m (ADCP)', 'red'), (3.0, '3.0m (GPS)', 'purple')]:
    ax3.axvline(x=threshold, color=color, linestyle=':', alpha=0.5)
    ax3.text(threshold, max(heights)*0.9, label, rotation=90, 
             fontsize=8, alpha=0.7, verticalalignment='top')

# Top instrument displacement vectors
ax4 = fig.add_subplot(224)
top_inst_data = displacement_results['Top_inst']
ax4.scatter(top_inst_data['dx'], top_inst_data['dy'], 
           c=mooring_tilt_angle, cmap='viridis', alpha=0.6, s=3)
ax4.set_title('Top Instrument Displacement Vectors\n(Color = Tilt Magnitude)')
ax4.set_xlabel('East Displacement (m)')
ax4.set_ylabel('North Displacement (m)')
ax4.grid(True, alpha=0.3)
ax4.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax4.axvline(x=0, color='k', linestyle='-', alpha=0.3)

# Add reference circles
for radius, label in [(1.0, '1m'), (2.0, '2m'), (3.0, '3m GPS')]:
    circle = plt.Circle((0, 0), radius, fill=False, color='red', 
                       linestyle='--', alpha=0.4, linewidth=1)
    ax4.add_patch(circle)
    ax4.text(radius*0.7, radius*0.7, label, color='red', fontsize=8, alpha=0.7)

plt.tight_layout()
plt.show()

print(f"✓ Clean plots created with corrected angles")
print(f"✓ {len(df_sample):,} points plotted (sampled from {len(df_accel_filtered):,} total)")
print(f"✓ Tilt range: {mooring_tilt_angle.min():.3f}° to {mooring_tilt_angle.max():.3f}°")
print(f"✓ Max displacement: {max(max_displacements.values()):.4f}m")

In [26]:
# ------------------------------------------------------------
# Time Series Plot of Tilt Data
# ------------------------------------------------------------

print("Creating detailed time series plots of tilt data...\n")

fig, axes = plt.subplots(4, 1, figsize=(15, 12))

# 1. Individual pitch and roll components
axes[0].plot(df_accel_filtered['Time'], df_accel_filtered['Pitch'], 
             'b-', alpha=0.7, linewidth=0.8, label='Pitch')
axes[0].plot(df_accel_filtered['Time'], df_accel_filtered['Roll'], 
             'r-', alpha=0.7, linewidth=0.8, label='Roll')
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].set_title('Pitch and Roll Components Over Time')
axes[0].set_ylabel('Angle (°)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Combined tilt magnitude
axes[1].plot(df_accel_filtered['Time'], mooring_tilt_angle, 
             'g-', alpha=0.8, linewidth=1.0)
axes[1].axhline(y=mooring_tilt_angle.mean(), color='red', linestyle='--', 
               label=f'Mean: {mooring_tilt_angle.mean():.3f}°', alpha=0.8)
axes[1].axhline(y=mooring_tilt_angle.quantile(0.95), color='orange', linestyle='--', 
               label=f'95th percentile: {mooring_tilt_angle.quantile(0.95):.3f}°', alpha=0.8)
axes[1].set_title('Combined Tilt Magnitude Over Time')
axes[1].set_ylabel('Tilt Angle (°)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Tilt direction (azimuth)
axes[2].plot(df_accel_filtered['Time'], tilt_azimuth, 
             'purple', alpha=0.7, linewidth=0.8)
axes[2].axhline(y=tilt_azimuth.mean(), color='red', linestyle='--', 
               label=f'Mean direction: {tilt_azimuth.mean():.1f}°', alpha=0.8)
axes[2].set_title('Tilt Direction (Azimuth) Over Time')
axes[2].set_ylabel('Direction (° from North)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 360)

# 4. Top instrument displacement
top_inst_data = displacement_results['Top_inst']
axes[3].plot(df_accel_filtered['Time'], top_inst_data['horizontal_displacement'], 
             'orange', alpha=0.8, linewidth=1.0)
axes[3].axhline(y=top_inst_data['horizontal_displacement'].mean(), color='red', linestyle='--', 
               label=f'Mean: {top_inst_data["horizontal_displacement"].mean():.4f}m', alpha=0.8)
axes[3].set_title('Top Instrument Horizontal Displacement Over Time')
axes[3].set_ylabel('Displacement (m)')
axes[3].set_xlabel('Time')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

# Format x-axis for all subplots
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print some statistics about the time series
print(f"Time Series Statistics:")
print(f"  Time span: {df_accel_filtered['Time'].min()} to {df_accel_filtered['Time'].max()}")
print(f"  Duration: {(df_accel_filtered['Time'].max() - df_accel_filtered['Time'].min()).days} days")
print(f"  Data points: {len(df_accel_filtered):,}")
print(f"  Sampling interval: ~{(df_accel_filtered['Time'].iloc[1] - df_accel_filtered['Time'].iloc[0]).total_seconds():.0f} seconds")

print(f"\nTilt Variability:")
print(f"  Pitch range: {df_accel_filtered['Pitch'].min():.3f}° to {df_accel_filtered['Pitch'].max():.3f}°")
print(f"  Roll range: {df_accel_filtered['Roll'].min():.3f}° to {df_accel_filtered['Roll'].max():.3f}°")
print(f"  Tilt magnitude range: {mooring_tilt_angle.min():.3f}° to {mooring_tilt_angle.max():.3f}°")
print(f"  Max displacement range: {top_inst_data['horizontal_displacement'].min():.4f}m to {top_inst_data['horizontal_displacement'].max():.4f}m")